# Fine-tune a Vocabulary Upgrade Model (GrammarLens)

Fine-tunes `t5-small` on the dataset produced by `generate_training_data.py`, so your app can suggest vocabulary upgrades with **its own local model** instead of calling the Gemini API at runtime.

**Before you start:** go to `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`, then Save. Training on CPU here will be painfully slow.

Run the cells in order, top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU. Set Runtime -> Change runtime type -> T4 GPU, then restart.")

## 2. Upload your dataset

Run this cell, then pick the `vocab_training_data.jsonl` file that `generate_training_data.py` created on your laptop.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("\nUploaded:", list(uploaded.keys()))

## 3. Load and inspect the data

Each row becomes a seq2seq pair:

- **input**: `upgrade: <word> context: <sentence>`
- **target**: `alt1, alt2, alt3`

The input format here must match exactly what `vocab_model.py` sends at inference time, or the model won't recognise the prompt shape it was trained on.

In [ ]:
import json

DATA_FILE = "vocab_training_data.jsonl"

rows = []
with open(DATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue

print(f"Loaded {len(rows)} raw rows")

pairs = []
for r in rows:
    word = str(r.get("original", "")).strip().lower()
    sentence = str(r.get("sentence", "")).strip()
    alts = r.get("alternatives", [])
    if not word or not sentence or not isinstance(alts, list) or not alts:
        continue
    pairs.append({
        "input_text": f"upgrade: {word} context: {sentence}",
        "target_text": ", ".join(str(a) for a in alts[:3]),
    })

print(f"Built {len(pairs)} training pairs\n")
print("Sample:")
for p in pairs[:3]:
    print("  IN :", p["input_text"])
    print("  OUT:", p["target_text"])
    print()

## 4. Split into train/validation

In [ ]:
import random
from datasets import Dataset

random.seed(42)
random.shuffle(pairs)

split = max(1, int(len(pairs) * 0.1))
val_pairs = pairs[:split]
train_pairs = pairs[split:]

train_ds = Dataset.from_list(train_pairs)
val_ds = Dataset.from_list(val_pairs)

print(f"Train: {len(train_ds)}   Validation: {len(val_ds)}")

## 5. Tokenize

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BASE_MODEL = "t5-small"   # small enough to run comfortably on your laptop afterwards
MAX_INPUT_LEN = 96
MAX_TARGET_LEN = 32

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

def preprocess(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

print("Tokenized.")
print(train_tok)

## 6. Train

Includes early stopping: training halts automatically once validation loss stops improving for 3 epochs straight, and the final model is always the best-performing checkpoint, not just whichever epoch happens to run last. This is the actual defense against overfitting on a modest dataset -- watch the `eval_loss` column in the printed table; if it starts creeping up while `loss` (training loss) keeps falling, that's overfitting, and early stopping is what catches it for you automatically.

In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

NUM_EPOCHS = 20          # an upper bound only -- early stopping below will stop earlier automatically
BATCH_SIZE = 8
LEARNING_RATE = 3e-4     # a typical, slightly high LR that suits T5 fine-tuning

collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="vocab_finetune_out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.05,               # a bit more regularization given the larger vocabulary
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",           # required for load_best_model_at_end to work
    save_total_limit=2,              # keep disk usage small -- only the 2 most recent checkpoints
    load_best_model_at_end=True,     # ALWAYS end up with the best-generalizing checkpoint, not just the last one
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    # Stops automatically if validation loss hasn't improved for 3 straight
    # epochs -- this is the actual overfitting guard, not just a fixed epoch count.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

print("\nBest validation loss achieved:", trainer.state.best_metric)


## 7. Try it out

Sanity-check the model before downloading it. Output should be a short comma-separated list of upgrade words.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

def suggest(word, sentence):
    prompt = f"upgrade: {word} context: {sentence}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=96).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=32, num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

tests = [
    ("good", "This is a good and important change."),
    ("help", "I hope this will help the problem."),
    ("big", "They announced a big update to the system."),
    ("make", "We need to make a decision by Friday."),
]

for w, s in tests:
    print(f"{w:10} -> {suggest(w, s)}")

## 8. Save and download

Produces `vocab_upgrade_model.zip`. Download it, unzip it into your project folder, and point `vocab_model.py` at it.

In [ ]:
SAVE_DIR = "vocab_upgrade_model"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

!zip -r -q vocab_upgrade_model.zip {SAVE_DIR}
!du -h vocab_upgrade_model.zip

from google.colab import files
files.download("vocab_upgrade_model.zip")